<a href="https://colab.research.google.com/github/Aylin-19-Johns/book-filtering-system/blob/main/Book_Filtering_system.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

books = pd.read_csv("/content/Books.csv", low_memory=False)
ratings = pd.read_csv("/content/Ratings.csv", low_memory=False)
users = pd.read_csv("/content/Users.csv", low_memory=False)

df = ratings.merge(books, on='ISBN')


# Filter out books with less than 50 ratings
book_counts = df['Book-Title'].value_counts()
popular_books = book_counts[book_counts >= 50].index
df = df[df['Book-Title'].isin(popular_books)]

# Filter out users who have rated less than 5 books
user_counts = df['User-ID'].value_counts()
active_users = user_counts[user_counts >= 5].index
df = df[df['User-ID'].isin(active_users)]

user_item_matrix = df.pivot_table(index='User-ID', columns='Book-Title', values='Book-Rating')

# Fill NaN with 0
user_item_matrix = user_item_matrix.fillna(0)

user_similarity = cosine_similarity(user_item_matrix)

user_similarity_df = pd.DataFrame(user_similarity,
                                  index=user_item_matrix.index,
                                  columns=user_item_matrix.index)


def recommend_books(user_id, n=5):

    # Check if user_id is in the matrix after filtering
    if user_id not in user_item_matrix.index:
        return f"User {user_id} not found or did not rate enough books to be included in the recommendation system."

    # Get similar users
    similar_users = user_similarity_df[user_id].sort_values(ascending=False)[1:6]

    # Get books rated by similar users
    similar_users_books = user_item_matrix.loc[similar_users.index]

    # Average ratings
    mean_ratings = similar_users_books.mean().sort_values(ascending=False)

    # Remove already rated books
    user_books = user_item_matrix.loc[user_id]
    unseen_books = mean_ratings[user_books == 0]

    # Top N recommendations
    recommendations = unseen_books.head(n)

    return recommendations

# Example user ID (change as needed)
# Ensure the user_id exists in the filtered matrix
if not user_item_matrix.empty:
    user_id = user_item_matrix.index[0]
    print(f"Top 5 Recommendations for User {user_id}:\n")
    print(recommend_books(user_id))
else:
    print("User-item matrix is empty after filtering. Cannot make recommendations.")

Top 5 Recommendations for User 99:

Book-Title
Generation X: Tales for an Accelerated Culture         1.6
Crazy for You                                          1.4
Microserfs                                             1.4
The Divine Secrets of the Ya-Ya Sisterhood: A Novel    1.0
Sun Also Rises                                         0.0
dtype: float64
